# Library

In [1]:
pip install requests beautifulsoup4 pandas

# Article

In [ ]:
import requests
from bs4 import BeautifulSoup

def debug_single_sinta(sinta_id):
    url = f"https://sinta.kemdiktisaintek.go.id/authors/profile/{sinta_id}/?view=scopus"
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    r = requests.get(url, headers=headers, timeout=30)
    print("Status Code:", r.status_code)

    soup = BeautifulSoup(r.text, "html.parser")

    articles = soup.select("div.ar-list-item")
    print("Jumlah artikel:", len(articles))
    print("-" * 50)

    for i, art in enumerate(articles, start=1):
        title = art.select_one("div.ar-title a")
        year = art.select_one("a.ar-year")
        cited = art.select_one("a.ar-cited")
        quartile = art.select_one("a.ar-quartile")
        journal = art.select_one("a.ar-pub")

        print(f"Artikel #{i}")
        print("Judul   :", title.text.strip() if title else "-")
        print("Tahun   :", year.text.strip() if year else "-")
        print("Cited   :", cited.text.strip() if cited else "-")
        print("Quartile:", quartile.text.strip() if quartile else "-")
        print("Jurnal  :", journal.text.strip() if journal else "-")
        print("-" * 50)

if __name__ == "__main__":
    debug_single_sinta(258567)


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

def single_sinta(sinta_id):

    views = ["scopus", "garuda", "googlescholar", "rama"]
    base = f"https://sinta.kemdiktisaintek.go.id/authors/profile/{sinta_id}/"

    results = []

    for view in views:

        url = f"{base}?view={view}"
        res = requests.get(url, headers=HEADERS, timeout=30)
        soup = BeautifulSoup(res.text, "html.parser")

        papers = soup.select(".ar-list-item")

        for p in papers:

            # Title + link
            title_tag = p.select_one(".ar-title a")
            title = title_tag.text.strip() if title_tag else None
            link = title_tag["href"] if title_tag else None

            # Year
            year_tag = p.select_one(".ar-year")
            year = year_tag.text.strip() if year_tag else None

            # Cited
            cited_tag = p.select_one(".ar-cited")
            cited = cited_tag.text.strip() if cited_tag else None

            # Quartile / Accreditation
            quartile_tag = p.select_one(".ar-quartile")
            quartile = quartile_tag.text.strip() if quartile_tag else None

            # Journal / Publisher
            pub_tag = p.select_one(".ar-pub")
            publisher = pub_tag.text.strip() if pub_tag else None

            # Authors
            author = None
            authors = None
            for a in p.find_all("a"):
                text = a.text.strip()
                if "Authors" in a.text:
                    authors = a.text.replace("Authors :", "").strip()
                
                if "Creator" in text:
                    author = text.replace("Creator :", "").strip()

                if "Author Order" in text:
                    authors = text.strip()

            results.append({
                "id_sinta": sinta_id,
                "source": view,
                "article_title": title,
                "author": author,
                "authors": authors,
                "publisher": publisher,
                "year": year,
                "cited": cited,
                "quartile": quartile,
                "url": link
            })

    return pd.DataFrame(results)

In [6]:
import pandas as pd

if __name__ == "__main__":
    data = pd.read_csv("../author/Tabel Authors Unikom Sinta.csv")
    all_results = []

    for sinta_id in data["id_sinta"]:

        print(f"Scraping SINTA ID : {sinta_id}")

        try:
            df = single_sinta(sinta_id)
            all_results.append(df)

        except Exception as e:
            print(f"Error pada {sinta_id} : {e}")

    # Gabungkan semua dataframe
    final_df = pd.concat(all_results, ignore_index=True)

    # Simpan CSV
    final_df.to_csv("Tabel Article Unikom Sinta.csv", index=False, encoding="utf-8")

    print("Scraping selesai, file tersimpan: sinta_publications.csv")

Scraping SINTA ID : 6754370
Scraping SINTA ID : 5985954
Scraping SINTA ID : 5999039
Scraping SINTA ID : 6655639
Scraping SINTA ID : 257962
Scraping SINTA ID : 257844
Scraping SINTA ID : 257836
Scraping SINTA ID : 258622
Scraping SINTA ID : 258595
Scraping SINTA ID : 258932
Scraping SINTA ID : 258587
Scraping SINTA ID : 258682
Scraping SINTA ID : 258935
Scraping SINTA ID : 258716
Scraping SINTA ID : 6665452
Scraping SINTA ID : 6003866
Scraping SINTA ID : 258641
Scraping SINTA ID : 6803098
Scraping SINTA ID : 6005094
Scraping SINTA ID : 258768
Scraping SINTA ID : 259067
Scraping SINTA ID : 258596
Scraping SINTA ID : 6666078
Scraping SINTA ID : 5999373
Scraping SINTA ID : 258648
Scraping SINTA ID : 258554
Scraping SINTA ID : 6005128
Scraping SINTA ID : 258739
Scraping SINTA ID : 6803105
Scraping SINTA ID : 258260
Scraping SINTA ID : 258586
Scraping SINTA ID : 6002971
Scraping SINTA ID : 258875
Scraping SINTA ID : 6003865
Scraping SINTA ID : 258601
Scraping SINTA ID : 258871
Scraping SINTA